In [1]:
# =====================================================================
# What this snippet teaches:
#   How K-Nearest Neighbors works internally: it stores the training
#   data, and at prediction time finds the K closest samples and takes
#   a majority vote. Built with NumPy only, no sklearn model.
#
# Project stage: Modeling (model implementation / baseline model)
#   Use it to understand or to build a quick baseline.
#
# Lesson summary (as comments):
#   - KNN is a lazy learner: fit() only stores data, predict() does work.
#   - Prediction = distance to all training points -> pick K smallest
#     -> majority vote (classification) or mean (regression).
#   - Distance depends on feature scale, so scale features BEFORE KNN.
#   - Small K = overfit (noisy boundary), large K = underfit (too smooth).
#   - Choose K with cross-validation, use odd K for binary problems.
#   - Cost: O(n * d) per query. Slow on big data without KD-tree/Ball-tree.
# =====================================================================

import numpy as np


class KNNClassifier:
    """Minimal KNN classifier using Euclidean distance."""

    def __init__(self, k: int = 5):
        if k < 1:
            raise ValueError("k must be >= 1")
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> "KNNClassifier":
        # Lazy learner: nothing is learned here, we just memorize the data
        if self.k > len(X):
            raise ValueError("k cannot be larger than the number of samples")
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def _predict_one(self, x: np.ndarray):
        # 1) Euclidean distance from x to every training sample (vectorized)
        distances = np.sqrt(((self.X_train - x) ** 2).sum(axis=1))

        # 2) Indices of the k smallest distances
        nearest_idx = np.argsort(distances)[: self.k]

        # 3) Majority vote among the neighbors' labels
        neighbor_labels = self.y_train[nearest_idx]
        labels, counts = np.unique(neighbor_labels, return_counts=True)
        return labels[np.argmax(counts)]

    def predict(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_one(x) for x in X])


def main() -> None:
    # TODO: load your dataset here
    # X, y = ...

    # TODO: split into train / test (use stratify=y for classification)
    # X_train, X_test, y_train, y_test = ...

    # TODO: scale features. Fit the scaler on TRAIN only, then transform both
    # scaler = StandardScaler().fit(X_train)
    # X_train, X_test = scaler.transform(X_train), scaler.transform(X_test)

    # TODO: choose k via cross-validation instead of guessing
    knn = KNNClassifier(k=5)
    # knn.fit(X_train, y_train)
    # y_pred = knn.predict(X_test)

    # TODO: evaluate (accuracy, and per-class metrics if data is imbalanced)
    # print(accuracy_score(y_test, y_pred))


if __name__ == "__main__":
    main()


# =====================================================================
# Engineering red flags :
#   - Features not scaled: the largest-range feature dominates distances
#   - Scaler fit on the full dataset (data leakage into the test set)
#   - NaN values in X: distances become NaN and neighbors are garbage
#   - Choosing k by looking at test accuracy (tune with CV on train only)
#   - Even k on a binary problem (tie votes)
#   - Large datasets (100k+ rows) with brute force: predict gets very slow
#   - Many irrelevant or high-dimensional features (curse of dimensionality)
#   - Imbalanced classes: the majority class wins most votes, so check
#     precision/recall, not just accuracy
#   - Categorical features encoded as arbitrary integers: distance on
#     them is meaningless
#   - Real projects: use sklearn's KNeighborsClassifier (KD-tree/Ball-tree);
#     this from-scratch version is for learning and baselines
# =====================================================================